In [7]:
import pandas as pd

# 2. Load the dataset
# Ensure leads.csv is in the same directory as this notebook
df = pd.read_csv("leads.csv")

# 3. Preview the first 5 rows and basic structure
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (9160, 20)


,lead_id,created_at,source,city,area,property_type,budget_pkr_lac,bedrooms,first_response_minutes,calls_made,total_call_seconds,whatsapp_replies,site_visits,agent_experience_years,is_overseas,referred_by_existing_client,has_financing_approved,token_amount_received_pkr,crm_record_hash,converted
0,MGC-104067,2025-09-30 19:35:11,Facebook Ads,Rawalpindi,Bahria Town,Farmhouse,801.0,1.0,12.0,2,72.0,4,1,4.7,0,0,1,0.0,8637176927,0
1,MGC-108870,2024-05-25 14:40:03,Instagram,Islamabad,Bahria Town,Commercial Shop,140.0,NaN,97.0,1,57.0,0,1,2.0,0,0,0,0.0,5937285284,0
2,MGC-101529,2024-07-30 02:06:11,Facebook Ads,Faisalabad,DHA,Penthouse,325.0,1.0,38.0,0,0.0,2,0,0.2,1,0,1,0.0,9807118704,0
3,MGC-108073,2025-05-08 22:01:20,Google Search,Islamabad,B-17,Apartment,85.0,3.0,15.0,1,138.0,3,0,NaN,1,0,1,0.0,6076182026,0
4,MGC-107878,2025-01-26 23:14:14,Instagram,Islamabad,Top City,Plot,117.0,NaN,27.0,2,54.0,2,0,6.3,0,0,0,0.0,4881826037,0


In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9160 entries, 0 to 9159
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   lead_id                      9160 non-null   str    
 1   created_at                   9160 non-null   str    
 2   source                       9160 non-null   str    
 3   city                         9160 non-null   str    
 4   area                         8683 non-null   str    
 5   property_type                9160 non-null   str    
 6   budget_pkr_lac               8876 non-null   float64
 7   bedrooms                     5558 non-null   float64
 8   first_response_minutes       8984 non-null   float64
 9   calls_made                   9160 non-null   int64  
 10  total_call_seconds           9160 non-null   float64
 11  whatsapp_replies             9160 non-null   int64  
 12  site_visits                  9160 non-null   int64  
 13  agent_experience_years       

In [11]:
import numpy as np

# 1. City Normalization Routine
def normalize_city(city):
    if pd.isna(city):
        return "Unknown"
    c = str(city).strip().upper()
    
    # Map common Pakistani city variations & abbreviations
    if c in ["ISB", "ISLAMABAD"]:
        return "Islamabad"
    if c in ["RWP", "RAWALPINDI"]:
        return "Rawalpindi"
    if c in ["KHI", "KARACHI"]:
        return "Karachi"
    if c in ["LHR", "LAHORE"]:
        return "Lahore"
    if c in ["PEW", "PESHAWAR"]:
        return "Peshawar"
    if c in ["MUX", "MULTAN"]:
        return "Multan"
    
    return c.title()

# 2. Apply cleaning to the DataFrame
df["city_clean"] = df["city"].apply(normalize_city)
df["source_clean"] = df["source"].fillna("Unknown").str.strip().str.lower()



# 4. Convert timestamp and sort chronologically for time-based splitting later
df["created_at"] = pd.to_datetime(df["created_at"])
df = df.sort_values("created_at").reset_index(drop=True)

# Inspect cleaned distributions
print("--- Top Normalized Cities ---")
print(df["city_clean"].value_counts().head(10))
print("\n--- Cleaned Lead Sources ---")
print(df["source_clean"].value_counts().head(10))

--- Top Normalized Cities ---
city_clean
Islamabad     2702
Rawalpindi    1981
Lahore        1375
Karachi        987
Peshawar       645
Faisalabad     513
Multan         389
Gujranwala     295
Abbottabad     273
Name: count, dtype: int64

--- Cleaned Lead Sources ---
source_clean
facebook ads         2366
property portal      1812
google search        1460
instagram            1007
referral              730
walk-in               610
whatsapp campaign     548
expo stall            345
billboard             282
Name: count, dtype: int64


In [13]:
import numpy as np

# 1. Deduplication Strategy
initial_count = len(df)

# Create a temporary sorting flag: Converted leads (1) come before unconverted leads (0)
df["_temp_priority"] = df["converted"].astype(int)

# Build a composite identity key from existing available text columns
if "first_name" in df.columns and "last_name" in df.columns:
    df["composite_id"] = (
        df["first_name"].fillna("").str.strip().str.lower() + "_" +
        df["last_name"].fillna("").str.strip().str.lower() + "_" +
        df["city_clean"].str.lower()
    )
    # Replace empty/blank matches so we don't drop leads missing names
    df["composite_id"] = df["composite_id"].replace("__", np.nan)
    
    # Sort: Converted leads first, then newest creation timestamp
    df = df.sort_values(
        by=["composite_id", "_temp_priority", "created_at"],
        ascending=[True, False, False]
    )
    
    # Retain 1 canonical winner per composite identity group
    valid_mask = df["composite_id"].notna()
    df_valid = df[valid_mask].drop_duplicates(subset=["composite_id"], keep="first")
    df = pd.concat([df_valid, df[~valid_mask]])

# Clean up the temporary column
df = df.drop(columns=["_temp_priority"])

# Re-sort strictly by creation date for clean out-of-time evaluation
df = df.sort_values("created_at").reset_index(drop=True)
print(f"Deduplicated dataset: Removed {initial_count - len(df)} duplicate rows ({len(df)} remaining).")

# 2. Define Features & Target
feature_cols = ["source_clean", "city_clean"]
X = df[feature_cols]
y = df["converted"].astype(int)

# 3. Time-Based Split (80% Historical Train, 20% Out-Of-Time Test)
split_idx = int(len(df) * 0.80)

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

train_start, train_end = df["created_at"].iloc[0], df["created_at"].iloc[split_idx - 1]
test_start, test_end = df["created_at"].iloc[split_idx], df["created_at"].iloc[-1]

print("\n--- Time-Based Split Summary ---")
print(f"Train Set: {len(X_train)} rows | Date Range: {train_start.strftime('%Y-%m-%d')} to {train_end.strftime('%Y-%m-%d')} | Conversion Rate: {y_train.mean():.2%}")
print(f"Test Set:  {len(X_test)} rows | Date Range: {test_start.strftime('%Y-%m-%d')} to {test_end.strftime('%Y-%m-%d')} | Conversion Rate: {y_test.mean():.2%}")

Deduplicated dataset: Removed 0 duplicate rows (9160 remaining).

--- Time-Based Split Summary ---
Train Set: 7328 rows | Date Range: 2024-01-01 to 2025-05-28 | Conversion Rate: 6.67%
Test Set:  1832 rows | Date Range: 2025-05-28 to 2025-10-01 | Conversion Rate: 7.91%


In [16]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

# 1. Build Preprocessing Pipeline
categorical_features = ["source_clean", "city_clean"]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat", 
            OneHotEncoder(handle_unknown="ignore", sparse_output=False), 
            categorical_features
        )
    ]
)

# 2. Define Model Pipeline (Logistic Regression Baseline)
pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(random_state=42, max_iter=1000))
    ]
)

# 3. Fit Pipeline on Historical Training Data
pipeline.fit(X_train, y_train)

# 4. Predict Conversion Probabilities on Out-Of-Time Test Data
# [:, 1] extracts the probability of target class (converted = 1)
test_probs = pipeline.predict_proba(X_test)[:, 1]

# Create a scoring DataFrame for evaluation
eval_df = X_test.copy()
eval_df["actual_converted"] = y_test.values
eval_df["predicted_score"] = test_probs

# 5. Evaluate Precision@100 (Top 100 Scored Leads)
top_100_leads = eval_df.sort_values(by="predicted_score", ascending=False).head(100)
top_100_conversions = top_100_leads["actual_converted"].sum()
top_100_precision = (top_100_conversions / 100) * 100

# 6. Baseline Comparisons (Random / No-Skill Calling)
test_overall_conversion_rate = y_test.mean()
random_100_conversions = round(test_overall_conversion_rate * 100, 1)

print("=" * 60)
print("             BASELINE MODEL EVALUATION RESULTS             ")
print("=" * 60)
print(f"Total Test Set Size:               {len(y_test)} leads")
print(f"Overall Test Conversion Rate:      {test_overall_conversion_rate:.2%}")
print("-" * 60)
print(f"Random Calling (Top 100 Leads):    ~{random_100_conversions} conversions")
print(f"Model-Scored Calling (Top 100):    {top_100_conversions} conversions")
print(f"Top 100 Precision (Precision@100): {top_100_precision:.1f}%")
print(f"Lift Over Baseline:               {top_100_conversions / max(random_100_conversions, 1):.2f}x")
print("=" * 60)

             BASELINE MODEL EVALUATION RESULTS             
Total Test Set Size:               1832 leads
Overall Test Conversion Rate:      7.91%
------------------------------------------------------------
Random Calling (Top 100 Leads):    ~7.9 conversions
Model-Scored Calling (Top 100):    25 conversions
Top 100 Precision (Precision@100): 25.0%
Lift Over Baseline:               3.16x


In [20]:
new_lead = pd.DataFrame([{
    'source_clean': 'facebook',
    'city_clean': 'Abbottabad'
}])

# Pipeline calculates conversion probability:
score = pipeline.predict_proba(new_lead)[0][1]
print(f"Lead Conversion Probability: {score:.1%}")

Lead Conversion Probability: 6.0%


In [21]:
import joblib

# Save the trained pipeline to disk
joblib.dump(pipeline, "lead_scoring_model.joblib")
print("Model saved successfully as lead_scoring_model.joblib")

Model saved successfully as lead_scoring_model.joblib
